In [1]:
import torch
path = "tsanet_model_android_cpu.pt"
try:
    m = torch.jit.load(path)
    print("Loaded as TorchScript. Example forward output shape:")
    x = torch.randn(1, 32, 1)
    print(m(x).shape)
except Exception as e:
    print("Not a TorchScript file:", e)


Loaded as TorchScript. Example forward output shape:
torch.Size([1, 7])


In [7]:
import torch, numpy as np, json

model = torch.jit.load("tsanet_model_android_cpu.pt")
model.eval()

# Load mapping
label_map = json.load(open("label_map.json"))

# Dummy 32 features
x = np.random.rand(32).astype(np.float32)
x_t = torch.tensor(x).view(1, 32, 1)

with torch.no_grad():
    out = model(x_t)
    probs = torch.softmax(out, dim=-1).numpy()[0]
    pred = int(np.argmax(probs))

print("Predicted:", label_map[str(pred)], "with confidence", probs[pred])


Predicted: DoS with confidence 0.5783262


In [8]:
pip install scapy joblib numpy pandas torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 22.0 MB/s eta 0:00:00


In [ ]:
#!/usr/bin/env python3
"""
live_flow_predict.py

Capture network packets on a laptop, aggregate flow-level features (32 features),
load a TorchScript model, optionally apply a saved scaler (joblib), and predict.

Usage:
    pip install scapy joblib numpy pandas torch
    sudo python3 live_flow_predict.py --iface eth0 \
        --model tsanet_model_android_cpu.pt --label_map label_map.json --scaler scaler.joblib

If you don't have scaler.joblib, omit --scaler (script will still run but results may be invalid).
"""

import argparse
import time
import json
from collections import defaultdict, deque
import threading
import numpy as np
import joblib
from scapy.all import sniff, IP, TCP, UDP
import torch

# ---------------------------
# The exact feature order you provided
FEATURE_ORDER = [
    'Flow Duration', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Std',
    'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
    'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
    'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd Packets/s',
    'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance',
    'FIN Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'Average Packet Size',
    'Avg Bwd Segment Size', 'Init_Win_bytes_forward', 'Active Mean', 'Active Min',
    'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'
]
# ---------------------------

# Per-flow data structure
# Key by 5-tuple: (src, sport, dst, dport, proto)
def flow_key(pkt):
    if IP not in pkt:
        return None
    ip = pkt[IP]
    proto = "TCP" if TCP in pkt else ("UDP" if UDP in pkt else str(pkt.proto))
    sport = pkt.sport if hasattr(pkt, 'sport') else 0
    dport = pkt.dport if hasattr(pkt, 'dport') else 0
    return (ip.src, int(sport), ip.dst, int(dport), proto)

class FlowState:
    def __init__(self, first_ts):
        self.start_ts = first_ts
        self.last_ts = first_ts
        self.pkt_times = []       # list of timestamps
        self.pkt_lengths = []     # list of lengths
        self.fwd_lengths = []     # forward direction lengths (src->dst)
        self.bwd_lengths = []     # backward direction lengths (dst->src)
        self.flags_counts = {'F':0,'P':0,'A':0,'S':0,'R':0}
        self.init_win_forward = None
        self.fwd_pkt_count = 0
        self.bwd_pkt_count = 0
        # For IATs
        self.ia_times = []        # all inter-arrival times
        self.fwd_iats = []
        self.bwd_iats = []
        # For active/idle calculations: maintain timestamps, then compute bursts
        self.packet_ts_sorted = []

    def add_packet(self, ts, length, direction, tcp_flags=None, tcp_win=None):
        # direction: 'fwd' or 'bwd' relative to flow key ordering
        if self.pkt_times:
            iat = ts - self.pkt_times[-1]
            self.ia_times.append(iat)
            if direction == 'fwd':
                self.fwd_iats.append(iat)
            else:
                self.bwd_iats.append(iat)
        self.pkt_times.append(ts)
        self.pkt_lengths.append(length)
        self.packet_ts_sorted.append(ts)
        if direction == 'fwd':
            self.fwd_lengths.append(length)
            self.fwd_pkt_count += 1
            if self.init_win_forward is None and tcp_win is not None:
                self.init_win_forward = tcp_win
        else:
            self.bwd_lengths.append(length)
            self.bwd_pkt_count += 1

        if tcp_flags:
            # tcp_flags is an int or string; handle both
            try:
                flags_int = int(tcp_flags)
                # scapy-style: numeric bits; but if provided as string, handle below
                # This numeric mapping is not required; we instead use scapy TCP.sprintf if available earlier
            except Exception:
                flags_int = None

            # If tcp_flags already parsed as string like 'S', 'A', etc. handle:
            if isinstance(tcp_flags, str):
                for ch in tcp_flags:
                    if ch == 'F': self.flags_counts['F'] += 1
                    if ch == 'P': self.flags_counts['P'] += 1
                    if ch == 'A': self.flags_counts['A'] += 1
                    if ch == 'S': self.flags_counts['S'] += 1
                    if ch == 'R': self.flags_counts['R'] += 1

    def compute_active_idle(self):
        # compute bursts (active) and idle gaps between bursts
        if not self.packet_ts_sorted:
            return {'active_mean':0,'active_min':0,'idle_mean':0,'idle_std':0,'idle_max':0,'idle_min':0}
        ts = sorted(self.packet_ts_sorted)
        # consider a burst if consecutive packets are within 1 second; else split bursts
        bursts = []
        curr_start = ts[0]
        curr_last = ts[0]
        for t in ts[1:]:
            if t - curr_last <= 1.0:  # within 1s -> same active burst
                curr_last = t
            else:
                bursts.append((curr_start, curr_last))
                curr_start = t
                curr_last = t
        bursts.append((curr_start, curr_last))
        active_durations = [b[1]-b[0]+0.000001 for b in bursts]  # +tiny to avoid 0
        idle_durations = []
        for i in range(1, len(bursts)):
            idle_durations.append(bursts[i][0] - bursts[i-1][1])
        # stats
        import math
        def mean(xs): return float(sum(xs)/len(xs)) if xs else 0.0
        def std(xs):
            if not xs: return 0.0
            m = mean(xs)
            return float((sum((x-m)**2 for x in xs)/len(xs))**0.5)
        result = {
            'active_mean': mean(active_durations),
            'active_min': min(active_durations) if active_durations else 0.0,
            'idle_mean': mean(idle_durations),
            'idle_std': std(idle_durations),
            'idle_max': max(idle_durations) if idle_durations else 0.0,
            'idle_min': min(idle_durations) if idle_durations else 0.0
        }
        return result

# Global flow table
flows = defaultdict(lambda: None)
flows_lock = threading.Lock()
WINDOW = 1.0  # seconds aggregation window; you can increase to 2.0 for more stable features

# Model & scaler placeholders (loaded later)
MODEL = None
SCALER = None
LABEL_MAP = None

def extract_features_from_flow(key, fstate, now_ts):
    # Ensure we compute using same order as FEATURE_ORDER
    # Many derived features:
    pkt_count = len(fstate.pkt_times)
    if pkt_count == 0:
        return None
    flow_duration = float(fstate.last_ts - fstate.start_ts) if fstate.last_ts and fstate.start_ts else 0.0
    bwd_arr = np.array(fstate.bwd_lengths) if fstate.bwd_lengths else np.array([])
    bwd_max = float(bwd_arr.max()) if bwd_arr.size>0 else 0.0
    bwd_mean = float(bwd_arr.mean()) if bwd_arr.size>0 else 0.0
    bwd_std = float(bwd_arr.std(ddof=0)) if bwd_arr.size>0 else 0.0

    iats = np.array(fstate.ia_times) if fstate.ia_times else np.array([])
    flow_iat_mean = float(iats.mean()) if iats.size>0 else 0.0
    flow_iat_std = float(iats.std(ddof=0)) if iats.size>0 else 0.0
    flow_iat_max = float(iats.max()) if iats.size>0 else 0.0
    flow_iat_min = float(iats.min()) if iats.size>0 else 0.0

    fwd_iats = np.array(fstate.fwd_iats) if fstate.fwd_iats else np.array([])
    fwd_iat_total = float(fwd_iats.sum()) if fwd_iats.size>0 else 0.0
    fwd_iat_mean = float(fwd_iats.mean()) if fwd_iats.size>0 else 0.0
    fwd_iat_std = float(fwd_iats.std(ddof=0)) if fwd_iats.size>0 else 0.0
    fwd_iat_max = float(fwd_iats.max()) if fwd_iats.size>0 else 0.0

    bwd_iats = np.array(fstate.bwd_iats) if fstate.bwd_iats else np.array([])
    bwd_iat_mean = float(bwd_iats.mean()) if bwd_iats.size>0 else 0.0
    bwd_iat_std = float(bwd_iats.std(ddof=0)) if bwd_iats.size>0 else 0.0
    bwd_iat_max = float(bwd_iats.max()) if bwd_iats.size>0 else 0.0

    bwd_packets_per_s = float(fstate.bwd_pkt_count / max(flow_duration, 1e-9))

    pkt_arr = np.array(fstate.pkt_lengths) if fstate.pkt_lengths else np.array([])
    max_pkt_len = float(pkt_arr.max()) if pkt_arr.size>0 else 0.0
    pkt_mean = float(pkt_arr.mean()) if pkt_arr.size>0 else 0.0
    pkt_std = float(pkt_arr.std(ddof=0)) if pkt_arr.size>0 else 0.0
    pkt_var = float(pkt_arr.var(ddof=0)) if pkt_arr.size>0 else 0.0

    fin_cnt = int(fstate.flags_counts.get('F',0))
    psh_cnt = int(fstate.flags_counts.get('P',0))
    ack_cnt = int(fstate.flags_counts.get('A',0))
    avg_pkt_size = float(pkt_arr.mean()) if pkt_arr.size>0 else 0.0

    avg_bwd_seg_size = float(np.array(fstate.bwd_lengths).mean()) if fstate.bwd_lengths else 0.0
    init_win_forward = float(fstate.init_win_forward) if fstate.init_win_forward is not None else 0.0

    ai = fstate.compute_active_idle()
    active_mean = float(ai['active_mean'])
    active_min = float(ai['active_min'])
    idle_mean = float(ai['idle_mean'])
    idle_std = float(ai['idle_std'])
    idle_max = float(ai['idle_max'])
    idle_min = float(ai['idle_min'])

    # Build features list according to FEATURE_ORDER
    feat_map = {
        'Flow Duration': flow_duration,
        'Bwd Packet Length Max': bwd_max,
        'Bwd Packet Length Mean': bwd_mean,
        'Bwd Packet Length Std': bwd_std,
        'Flow IAT Mean': flow_iat_mean,
        'Flow IAT Std': flow_iat_std,
        'Flow IAT Max': flow_iat_max,
        'Flow IAT Min': flow_iat_min,
        'Fwd IAT Total': fwd_iat_total,
        'Fwd IAT Mean': fwd_iat_mean,
        'Fwd IAT Std': fwd_iat_std,
        'Fwd IAT Max': fwd_iat_max,
        'Bwd IAT Mean': bwd_iat_mean,
        'Bwd IAT Std': bwd_iat_std,
        'Bwd IAT Max': bwd_iat_max,
        'Bwd Packets/s': bwd_packets_per_s,
        'Max Packet Length': max_pkt_len,
        'Packet Length Mean': pkt_mean,
        'Packet Length Std': pkt_std,
        'Packet Length Variance': pkt_var,
        'FIN Flag Count': fin_cnt,
        'PSH Flag Count': psh_cnt,
        'ACK Flag Count': ack_cnt,
        'Average Packet Size': avg_pkt_size,
        'Avg Bwd Segment Size': avg_bwd_seg_size,
        'Init_Win_bytes_forward': init_win_forward,
        'Active Mean': active_mean,
        'Active Min': active_min,
        'Idle Mean': idle_mean,
        'Idle Std': idle_std,
        'Idle Max': idle_max,
        'Idle Min': idle_min
    }

    features = [float(feat_map.get(name, 0.0)) for name in FEATURE_ORDER]
    return features

# Packet handler
def packet_handler(pkt):
    try:
        k = flow_key(pkt)
        if k is None:
            return
        now = time.time()
        # direction: define forward as src==k[0] and sport==k[1] when packet captured
        ip = pkt[IP]
        proto = "TCP" if TCP in pkt else ("UDP" if UDP in pkt else str(pkt.proto))
        sport = int(pkt.sport) if hasattr(pkt, 'sport') else 0
        dport = int(pkt.dport) if hasattr(pkt, 'dport') else 0
        # match direction: if src==k[0] and sport==k[1] => forward else backward
        direction = 'fwd' if (ip.src == k[0] and sport == k[1]) else 'bwd'

        length = len(pkt)
        tcp_flags = None
        tcp_win = None
        if TCP in pkt:
            # scapy exposes flags as an int or string; use str() for counting letters
            tcp_flags = pkt[TCP].flags.__str__() if hasattr(pkt[TCP].flags, '__str__') else str(pkt[TCP].flags)
            try:
                tcp_win = int(pkt[TCP].window)
            except Exception:
                tcp_win = None

        with flows_lock:
            if flows[k] is None:
                flows[k] = FlowState(now)
            fs = flows[k]
            fs.last_ts = now
            fs.add_packet(ts=now, length=length, direction=direction, tcp_flags=tcp_flags, tcp_win=tcp_win)

# Background thread to periodically scan flows and emit features
def periodic_emit(model, scaler, label_map, iface, print_every=1.0):
    last_check = time.time()
    while True:
        time.sleep(0.2)
        now = time.time()
        to_emit = []
        with flows_lock:
            for k, fs in list(flows.items()):
                # If a flow has been inactive for >= WINDOW (no packets in last WINDOW), or duration > WINDOW, emit
                if fs is None:
                    continue
                inactive = now - fs.last_ts
                duration = fs.last_ts - fs.start_ts if fs.start_ts and fs.last_ts else 0.0
                if inactive >= WINDOW or duration >= WINDOW:
                    feats = extract_features_from_flow(k, fs, now)
                    if feats is not None:
                        to_emit.append((k, feats))
                    # reset flow state for this key
                    flows[k] = None
        # send to model
        for (k, feats) in to_emit:
            # apply scaler if present
            x = np.array(feats, dtype=np.float32).reshape(1, -1)
            if scaler is not None:
                try:
                    x = scaler.transform(x)
                except Exception as e:
                    print("Warning: scaler.transform failed:", e)
            # reshape for model: (1, 32, 1)
            x_t = torch.from_numpy(x).float().view(1, 32, 1)
            with torch.no_grad():
                out = model(x_t)
                probs = torch.softmax(out, dim=-1).cpu().numpy().tolist()[0]
                pred = int(np.argmax(probs))
                label = label_map.get(str(pred), str(pred)) if label_map else str(pred)
            print(f"[{time.strftime('%H:%M:%S')}] Flow {k} -> {label} (p={probs[pred]:.3f})")
        # continue loop

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--iface', required=True, help='capture interface, e.g., eth0 or wlan0')
    parser.add_argument('--model', required=True, help='path to TorchScript model (.pt)')
    parser.add_argument('--scaler', required=False, help='optional path to scaler.joblib')
    parser.add_argument('--label_map', required=False, help='optional JSON file mapping label id -> name')
    parser.add_argument('--window', type=float, default=1.0, help='flow window in seconds (default 1.0)')
    args = parser.parse_args()

    global MODEL, SCALER, LABEL_MAP, WINDOW
    WINDOW = args.window

    print("Loading model", args.model)
    MODEL = torch.jit.load(args.model, map_location='cpu')
    MODEL.eval()
    # limit threads
    torch.set_num_threads(2)

    SCALER = None
    if args.scaler:
        try:
            SCALER = joblib.load(args.scaler)
            print("Loaded scaler:", args.scaler)
        except Exception as e:
            print("Failed to load scaler:", e)
            SCALER = None

    LABEL_MAP = None
    if args.label_map:
        try:
            LABEL_MAP = json.load(open(args.label_map))
            print("Loaded label_map:", args.label_map)
        except Exception as e:
            print("Failed to load label_map:", e)
            LABEL_MAP = None

    # Start background emitter thread
    t = threading.Thread(target=periodic_emit, args=(MODEL, SCALER, LABEL_MAP, args.iface), daemon=True)
    t.start()

    print("Starting sniff on interface", args.iface, "— press Ctrl+C to stop")
    try:
        sniff(iface=args.iface, prn=packet_handler, store=0)
    except KeyboardInterrupt:
        print("Stopping capture. Exiting.")
        return

if __name__ == '__main__':
    main()


SyntaxError: expected 'except' or 'finally' block (ipython-input-3639900708.py, line 283)

In [12]:
pip install scapy torch numpy

In [14]:
from scapy.all import sniff, IP, TCP, UDP
import torch
import numpy as np
import time
import json

# Load TorchScript model
model = torch.jit.load("tsanet_model_android_cpu.pt")
model.eval()

# Example label mapping (adjust to your dataset)
label_map = {
    0: "BENIGN",
    1: "Bot",
    2: "Brute Force",
    3: "DDoS",
    4: "DoS",
    5: "Heartbleed",
    6: "Infiltration",
    # If you trained only 7 classes, trim here
}

# Simple feature extractor from packet
def extract_features(pkt):
    features = []

    # Basic header features
    if IP in pkt:
        ip = pkt[IP]
        features.append(len(pkt))                 # packet length
        features.append(ip.ttl)                   # TTL
        features.append(int(ip.flags))            # flags
    else:
        features.extend([0,0,0])

    if TCP in pkt:
        tcp = pkt[TCP]
        features.append(tcp.sport)                # src port
        features.append(tcp.dport)                # dst port
        features.append(tcp.flags)                # TCP flags
    elif UDP in pkt:
        udp = pkt[UDP]
        features.append(udp.sport)
        features.append(udp.dport)
        features.append(0)  # no flags
    else:
        features.extend([0,0,0])

    # Add dummy values for missing flow features
    while len(features) < 32:
        features.append(0.0)

    return np.array(features[:32], dtype=np.float32).reshape(1, 32, 1)

# Prediction function
def predict(pkt):
    x = extract_features(pkt)
    x_tensor = torch.tensor(x)
    with torch.no_grad():
        out = model(x_tensor)
        probs = torch.softmax(out, dim=-1).numpy()[0]
        pred = int(np.argmax(probs))
    print(f"Predicted: {label_map.get(pred, 'Unknown')} ({probs[pred]:.4f})")

# Sniff live packets (change iface if needed)
print("🚦 Starting packet capture, press Ctrl+C to stop...")
sniff(iface="eth0", prn=predict, store=0)


🚦 Starting packet capture, press Ctrl+C to stop...


<Sniffed: TCP:0 UDP:0 ICMP:0 Other:0>

In [ ]:
    from scapy.all import sniff, IP, TCP, UDP
    import torch
    import numpy as np
    import time
    import json
    from collections import defaultdict

    # -----------------------
    # Load Model
    # -----------------------
    model = torch.jit.load("tsanet_model_android_cpu.pt")
    model.eval()

    label_map = {
        0: "BENIGN",
        1: "Bot",
        2: "Brute Force",
        3: "DDoS",
        4: "DoS",
        5: "Heartbleed",
        6: "Infiltration"
    }

    # 32 features order
    important_features = [
        'Flow Duration', 'Bwd Packet Length Max', 'Bwd Packet Length Mean', 'Bwd Packet Length Std',
        'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
        'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
        'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd Packets/s',
        'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance',
        'FIN Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'Average Packet Size',
        'Avg Bwd Segment Size', 'Init_Win_bytes_forward', 'Active Mean', 'Active Min',
        'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min'
    ]

    # -----------------------
    # Flow Data Structure
    # -----------------------
    flows = defaultdict(lambda: {
        "packets": [],
        "times": [],
        "start": None,
        "end": None
    })

    def extract_features(flow):
        pkts = flow["packets"]
        times = flow["times"]

        if not pkts:
            return None

        fwd_ip = pkts[0][IP].src
        fwd_pkts = [p for p in pkts if p[IP].src == fwd_ip]
        bwd_pkts = [p for p in pkts if p[IP].src != fwd_ip]

        # 1. Flow Duration
        flow_duration = (flow["end"] - flow["start"]) if flow["start"] and flow["end"] else 0

        # 2-4. Bwd Packet Length stats
        bwd_lengths = [len(p) for p in bwd_pkts]
        bwd_len_max = np.max(bwd_lengths) if bwd_lengths else 0
        bwd_len_mean = np.mean(bwd_lengths) if bwd_lengths else 0
        bwd_len_std = np.std(bwd_lengths) if bwd_lengths else 0

        # 5-8. Flow IAT stats
        if len(times) > 1:
            iat = np.diff(times)
            flow_iat_mean = np.mean(iat)
            flow_iat_std = np.std(iat)
            flow_iat_max = np.max(iat)
            flow_iat_min = np.min(iat)
        else:
            flow_iat_mean = flow_iat_std = flow_iat_max = flow_iat_min = 0

        # 9-12. Fwd IAT stats
        fwd_times = [p.time for p in fwd_pkts]
        if len(fwd_times) > 1:
            fwd_iat = np.diff(fwd_times)
            fwd_iat_total = np.sum(fwd_iat)
            fwd_iat_mean = np.mean(fwd_iat)
            fwd_iat_std = np.std(fwd_iat)
            fwd_iat_max = np.max(fwd_iat)
        else:
            fwd_iat_total = fwd_iat_mean = fwd_iat_std = fwd_iat_max = 0

        # 13-15. Bwd IAT stats
        bwd_times = [p.time for p in bwd_pkts]
        if len(bwd_times) > 1:
            bwd_iat = np.diff(bwd_times)
            bwd_iat_mean = np.mean(bwd_iat)
            bwd_iat_std = np.std(bwd_iat)
            bwd_iat_max = np.max(bwd_iat)
        else:
            bwd_iat_mean = bwd_iat_std = bwd_iat_max = 0

        # 16. Bwd Packets/s
        bwd_pkts_per_s = len(bwd_pkts) / flow_duration if flow_duration > 0 else 0

        # 17-20. Packet Length stats
        all_lengths = [len(p) for p in pkts]
        max_len = np.max(all_lengths) if all_lengths else 0
        mean_len = np.mean(all_lengths) if all_lengths else 0
        std_len = np.std(all_lengths) if all_lengths else 0
        var_len = np.var(all_lengths) if all_lengths else 0

        # 21-23. TCP flag counts
        fin_count = sum(1 for p in pkts if TCP in p and p[TCP].flags.F)
        psh_count = sum(1 for p in pkts if TCP in p and p[TCP].flags.P)
        ack_count = sum(1 for p in pkts if TCP in p and p[TCP].flags.A)

        # 24. Avg Packet Size
        avg_pkt_size = mean_len

        # 25. Avg Bwd Segment Size
        bwd_payloads = [len(p[TCP].payload) for p in bwd_pkts if TCP in p]
        avg_bwd_seg_size = np.mean(bwd_payloads) if bwd_payloads else 0

        # 26. Init_Win_bytes_forward
        init_win_fwd = fwd_pkts[0][TCP].window if fwd_pkts and TCP in fwd_pkts[0] else 0

        # 27-28. Active times (approx: time active during flow)
        if len(times) > 1:
            active_times = np.diff(times)
            active_mean = np.mean(active_times)
            active_min = np.min(active_times)
        else:
            active_mean = active_min = 0

        # 29-32. Idle times (gaps between packets)
        if len(times) > 1:
            idle_times = np.diff(times)
            idle_mean = np.mean(idle_times)
            idle_std = np.std(idle_times)
            idle_max = np.max(idle_times)
            idle_min = np.min(idle_times)
        else:
            idle_mean = idle_std = idle_max = idle_min = 0

        feats = {
            'Flow Duration': flow_duration,
            'Bwd Packet Length Max': bwd_len_max,
            'Bwd Packet Length Mean': bwd_len_mean,
            'Bwd Packet Length Std': bwd_len_std,
            'Flow IAT Mean': flow_iat_mean,
            'Flow IAT Std': flow_iat_std,
            'Flow IAT Max': flow_iat_max,
            'Flow IAT Min': flow_iat_min,
            'Fwd IAT Total': fwd_iat_total,
            'Fwd IAT Mean': fwd_iat_mean,
            'Fwd IAT Std': fwd_iat_std,
            'Fwd IAT Max': fwd_iat_max,
            'Bwd IAT Mean': bwd_iat_mean,
            'Bwd IAT Std': bwd_iat_std,
            'Bwd IAT Max': bwd_iat_max,
            'Bwd Packets/s': bwd_pkts_per_s,
            'Max Packet Length': max_len,
            'Packet Length Mean': mean_len,
            'Packet Length Std': std_len,
            'Packet Length Variance': var_len,
            'FIN Flag Count': fin_count,
            'PSH Flag Count': psh_count,
            'ACK Flag Count': ack_count,
            'Average Packet Size': avg_pkt_size,
            'Avg Bwd Segment Size': avg_bwd_seg_size,
            'Init_Win_bytes_forward': init_win_fwd,
            'Active Mean': active_mean,
            'Active Min': active_min,
            'Idle Mean': idle_mean,
            'Idle Std': idle_std,
            'Idle Max': idle_max,
            'Idle Min': idle_min
        }

        return feats

    def predict_from_flow(flow):
        feats = extract_features(flow)
        if feats is None: return
        vec = np.array([feats[f] for f in important_features], dtype=np.float32).reshape(1,32,1)
        x = torch.tensor(vec)
        with torch.no_grad():
            out = model(x)
            probs = torch.softmax(out, dim=-1).numpy()[0]
            pred = int(np.argmax(probs))
        print(f"🚨 Prediction: {label_map.get(pred, 'Unknown')} (conf={probs[pred]:.4f})")

    # -----------------------
    # Packet Handler
    # -----------------------
    def handle_packet(pkt):
        if IP in pkt and (TCP in pkt or UDP in pkt):
            flow_key = tuple(sorted([(pkt[IP].src, pkt.sport if TCP in pkt else pkt[UDP].sport),
                                    (pkt[IP].dst, pkt.dport if TCP in pkt else pkt[UDP].dport)]))
            f = flows[flow_key]
            f["packets"].append(pkt)
            f["times"].append(pkt.time)
            if not f["start"]: f["start"] = pkt.time
            f["end"] = pkt.time

            # If flow has >10 packets, try prediction
            if len(f["packets"]) >= 10:
                predict_from_flow(f)
                flows.pop(flow_key)

    # -----------------------
    # Start Sniffing
    # -----------------------
    print("🚦 Starting real-time IDS... press Ctrl+C to stop.")
    sniff(iface="eth0", prn=handle_packet, store=0)


🚦 Starting real-time IDS... press Ctrl+C to stop.
🚨 Prediction: BENIGN (conf=0.8604)
🚨 Prediction: BENIGN (conf=0.6664)
🚨 Prediction: Heartbleed (conf=0.4989)
🚨 Prediction: BENIGN (conf=0.9017)
🚨 Prediction: BENIGN (conf=0.8954)
🚨 Prediction: DoS (conf=0.4847)
🚨 Prediction: DoS (conf=0.7415)
🚨 Prediction: BENIGN (conf=0.8859)
🚨 Prediction: DoS (conf=0.5000)
🚨 Prediction: BENIGN (conf=0.9593)
🚨 Prediction: BENIGN (conf=0.9117)
🚨 Prediction: BENIGN (conf=0.9123)
🚨 Prediction: BENIGN (conf=0.5800)
🚨 Prediction: Heartbleed (conf=0.6001)
🚨 Prediction: Heartbleed (conf=0.4682)
🚨 Prediction: BENIGN (conf=0.5594)
🚨 Prediction: BENIGN (conf=0.8834)
🚨 Prediction: BENIGN (conf=0.8825)
🚨 Prediction: Heartbleed (conf=0.5531)
🚨 Prediction: BENIGN (conf=0.8912)
🚨 Prediction: BENIGN (conf=0.5482)
🚨 Prediction: BENIGN (conf=0.8099)
🚨 Prediction: BENIGN (conf=0.9201)
🚨 Prediction: Heartbleed (conf=0.7096)
🚨 Prediction: BENIGN (conf=0.8419)
🚨 Prediction: DoS (conf=0.7487)
🚨 Prediction: BENIGN (conf=0.890

<Sniffed: TCP:0 UDP:0 ICMP:0 Other:0>